# 6.32 — The Lottery Ticket Hypothesis

The lottery ticket hypothesis says that inside a randomly initialized dense neural network there may already be a much smaller **winning ticket**: a sparse subnetwork that trains well when we keep its original lucky initialization. In this lesson, you will build the mask, the reset, the pruning loop, and the stability checks from scratch in NumPy so the hypothesis is a concrete computation rather than a slogan.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build the lottery-ticket hypothesis one idea at a time. Run each cell in order and read the printed intermediate values — every mask, gradient, pruning decision, and reset is shown. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, matrix products, masks, and small scratch neural networks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for initializations and pruning demos.

### 1. A dense network starts as many random candidate paths

A tiny two-layer network is already a lottery: every input-to-hidden and hidden-to-output weight is a possible signal path. At initialization, most paths are not special individually, but some signs and magnitudes happen to line up with the data. The hypothesis is not that the dense model is unnecessary after training; it is that the dense random initialization may contain a sparse subnetwork that can train well if we keep the right weights and reset them to their original values.

In [ ]:
X_w = np.array([[-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0]])  # XOR-style inputs.
y_w = np.array([[0.0], [1.0], [1.0], [0.0]])  # nonlinear targets needing hidden units.
W1_0_w = np.array([[1.20, -0.70, 0.30, -1.10], [0.40, 1.00, -0.80, 0.60]])  # original lucky input weights.
b1_0_w = np.zeros(4)  # hidden bias initialization.
W2_0_w = np.array([[0.90], [-1.30], [0.50], [0.20]])  # original hidden-to-output weights.
b2_0_w = np.zeros(1)  # output bias initialization.
print("dense parameter count:", W1_0_w.size + W2_0_w.size)  # 8 + 4 = 12 weights.
print("first-layer shape:", W1_0_w.shape, "second-layer shape:", W2_0_w.shape)  # inspect connectivity.
assert W1_0_w.size + W2_0_w.size == 12  # concrete count for this network.

▶ What you'll see: the network has 12 trainable weights, each one a candidate edge that a mask could keep or remove.

In [ ]:
plt.figure(figsize=(5, 3))  # create a compact view of the initial first-layer weights.
plt.imshow(W1_0_w, cmap="coolwarm", aspect="auto")  # color shows sign and magnitude.
plt.colorbar(label="initial weight")  # add numeric scale.
plt.title("1: dense initialization W1")  # title the initialization heatmap.
plt.xlabel("hidden unit")  # columns are hidden units.
plt.ylabel("input feature")  # rows are input features.
plt.show()  # display the heatmap.

▶ What you'll see: positive and negative weights are scattered before training; pruning will later decide which edges mattered.

*Why it's done this way:* a dense random network gives optimization many possible paths. The lottery-ticket claim is about selecting a **subgraph of this original draw**, so we must save `W1_0_w` and `W2_0_w` before any training changes them.

### 2. Forward signal, gating, and probability are local computations

The mathematics block uses a two-input scratch pass to show the local arithmetic. An affine score combines inputs and weights, ReLU gates negative signals away, and a softmax-style comparison turns scores into relative probability. These pieces are small, but repeated across a deep network they decide whether signal and gradient survive.

In [ ]:
x_w = np.array([1.5, -0.5])  # the visible two-input example.
w_w = np.array([1.4, -0.7])  # two incoming weights for one hidden unit.
b_w = 0.5  # bias from the lesson arithmetic.
z_w = float(x_w @ w_w + b_w)  # affine signal.
h_w = max(0.0, z_w)  # ReLU gated signal.
print("affine z:", round(z_w, 3), "gated h:", round(h_w, 3))  # 2.95 survives ReLU.
assert round(z_w, 3) == 2.950 and round(h_w, 3) == 2.950  # lesson arithmetic.

▶ What you'll see: `1.4*1.5 + (-0.7)*(-0.5) + 0.5 = 2.95`, and ReLU keeps it because it is positive.

In [ ]:
baseline_w = 0.4  # comparison score.
exp_scores_w = np.exp(np.array([h_w, baseline_w]))  # exponentiate both scores.
prob_w = float(exp_scores_w[0] / exp_scores_w.sum())  # softmax probability for the lesson score.
print("exp values:", np.round(exp_scores_w, 3), "probability:", round(prob_w, 3))  # 0.928.
assert round(prob_w, 3) == 0.928  # concrete comparison number.
plt.figure(figsize=(4, 3))  # visualize the normalized comparison.
plt.bar(["ticket score", "baseline"], exp_scores_w / exp_scores_w.sum(), color=["teal", "gray"])  # probability mass.
plt.title("2: softmax comparison")  # title the probability plot.
plt.ylabel("probability")  # label probability scale.
plt.show()  # display the bar chart.

▶ What you'll see: the larger gated score receives about 92.8 percent of the two-way softmax mass.

*Why it's done this way:* the mask only makes sense through the function it preserves. Affine maps create signals, ReLU selects active paths, and softmax compares alternatives; if these local quantities are unstable, a sparse subnetwork inherits that instability.

### 3. A binary mask defines the candidate ticket

A lottery ticket is represented by a binary mask `m` with 1 for kept weights and 0 for removed weights. The subnetwork computes with `m * theta`, so removed weights contribute exactly zero while the surviving weights keep their original coordinates and shapes.

In [ ]:
scores_w = np.abs(W1_0_w)  # use magnitude as a simple saliency score for this first mask demo.
threshold_w = 0.75  # keep only weights with magnitude at least 0.75.
mask1_w = (scores_w >= threshold_w).astype(float)  # binary mask over W1.
W1_ticket_w = mask1_w * W1_0_w  # masked first-layer weights.
print("mask W1:\n", mask1_w.astype(int))  # inspect kept edges.
print("kept W1 weights:", int(mask1_w.sum()), "of", mask1_w.size)  # sparsity count.
assert int(mask1_w.sum()) == 4  # concrete kept count.

▶ What you'll see: four of eight first-layer weights survive the threshold, and the rest become zeros.

In [ ]:
full_pre_w = X_w @ W1_0_w + b1_0_w  # dense hidden pre-activations.
ticket_pre_w = X_w @ W1_ticket_w + b1_0_w  # masked hidden pre-activations.
print("dense first input pre-activation:", np.round(full_pre_w[0], 3))  # inspect dense signal.
print("ticket first input pre-activation:", np.round(ticket_pre_w[0], 3))  # inspect sparse signal.
plt.figure(figsize=(5, 3))  # visualize dense versus sparse first-layer matrix.
plt.imshow(W1_ticket_w, cmap="coolwarm", aspect="auto")  # zeros show pruned weights.
plt.colorbar(label="masked weight")  # add numeric scale.
plt.title("3: W1 after applying the mask")  # title heatmap.
plt.xlabel("hidden unit")  # hidden-unit columns.
plt.ylabel("input feature")  # input-feature rows.
plt.show()  # display heatmap.

▶ What you'll see: the sparse pre-activations differ because removed edges no longer carry signal.

*Why it's done this way:* the formula `f(x; m ⊙ θ0)` means the architecture is not re-randomized or reshaped; it is the same coordinate system with selected weights set to zero. That lets us ask whether the original lucky coordinates mattered.

### 4. Training updates weights, but the ticket test resets survivors

Training moves parameters by repeated small gradient steps. The lottery-ticket test is special: after identifying a mask from a trained network, we reset the surviving weights back to their **original initialization** and train only those survivors. That separates “good sparse architecture” from “large weights discovered late in training.”

In [ ]:
theta_w = 2.0  # one scalar parameter.
eta_w = 0.07  # learning rate.
grad_w = 1.8  # gradient of the loss with respect to theta.
theta_new_w = theta_w - eta_w * grad_w  # gradient descent update.
print("updated theta:", round(theta_new_w, 3))  # 1.874.
assert round(theta_new_w, 3) == 1.874  # lesson update number.

▶ What you'll see: a single scalar moves by 0.126, illustrating that training is many small reliable nudges.

In [ ]:
trained_W1_w = W1_0_w - 0.10 * np.sign(W1_0_w)  # pretend training changed every surviving coordinate.
lottery_reset_W1_w = mask1_w * W1_0_w  # reset survivors to the original lucky initialization.
late_pruned_W1_w = mask1_w * trained_W1_w  # keep trained values instead of resetting.
print("original kept sum:", round(float(lottery_reset_W1_w.sum()), 3))  # original ticket values.
print("late-pruned kept sum:", round(float(late_pruned_W1_w.sum()), 3))  # trained values differ.
assert round(float(lottery_reset_W1_w.sum()), 3) == 0.300  # concrete reset sum.

▶ What you'll see: resetting and late pruning keep the same edges but not the same numerical weights.

*Why it's done this way:* if the reset ticket trains well, the sparse subnetwork was already encoded by the original draw plus mask. If only the late-pruned weights work, then pruning found a compressed trained model, which is useful but not the exact lottery-ticket claim.

### 5. Iterative magnitude pruning finds a sparse mask

In practice, a common recipe is train dense, prune the smallest-magnitude weights, reset the survivors to their initial values, and repeat. Magnitude is a heuristic for “this weight ended training near zero, so perhaps the function does not rely on it.” Iteration matters because removing too much at once can destroy paths before the remaining network has adapted.

In [ ]:
trained_abs_w = np.array([0.90, 0.05, 0.42, 0.12, 1.10, 0.31, 0.08, 0.70, 0.22, 0.60])  # pretend trained magnitudes.
keep_fraction_w = 0.60  # keep the largest 60 percent after this pruning round.
cut_w = np.sort(trained_abs_w)[int((1 - keep_fraction_w) * trained_abs_w.size)]  # magnitude cutoff.
mask_flat_w = (trained_abs_w >= cut_w).astype(int)  # keep large magnitudes.
print("cutoff:", round(float(cut_w), 3), "kept:", int(mask_flat_w.sum()), "of", mask_flat_w.size)  # inspect pruning result.
assert int(mask_flat_w.sum()) == 6  # concrete kept count.

▶ What you'll see: the four smallest magnitudes are removed, leaving six surviving weights.

In [ ]:
rounds_w = np.arange(6)  # pruning rounds from dense to sparse.
remaining_w = 0.8 ** rounds_w  # keep 80 percent of remaining weights each round.
print("remaining fractions:", np.round(remaining_w, 3))  # multiplicative sparsity.
assert round(float(remaining_w[-1]), 3) == 0.328  # after five rounds about 32.8 percent remain.
plt.figure(figsize=(5, 3))  # create a pruning schedule plot.
plt.plot(rounds_w, remaining_w, marker="o", color="purple")  # show multiplicative decay.
plt.title("5: iterative pruning schedule")  # title curve.
plt.xlabel("pruning round")  # label rounds.
plt.ylabel("fraction of weights remaining")  # label remaining fraction.
plt.ylim(0, 1.05)  # keep scale interpretable.
plt.show()  # display the curve.

▶ What you'll see: pruning 20 percent repeatedly produces a smooth sparsity path rather than one abrupt cliff.

*Why it's done this way:* magnitude pruning is not a theorem; it is a practical selection rule. Iteration makes the search gentler, because each round asks which weights remain small after the network has had a chance to redistribute function.

### 6. Scale, normalization, and memory decide whether tickets are useful

A sparse ticket is valuable only if it still trains stably and saves resources. Normalization shows whether a signal is unusually large relative to its batch scale, and memory bookkeeping shows why sparse subnetworks matter on hardware. The lesson's normalized value and 1 KB activation block are small versions of the same large-scale accounting.

In [ ]:
value_w = 2.95  # gated signal from the scratch pass.
mean_w = 1.0  # reference mean.
var_w = 0.25  # reference variance.
eps_w = 1e-5  # numerical stability term.
normalized_w = (value_w - mean_w) / np.sqrt(var_w + eps_w)  # batch/layer normalization arithmetic.
print("normalized value:", round(float(normalized_w), 3))  # 3.9.
assert round(float(normalized_w), 3) == 3.900  # concrete lesson number.

▶ What you'll see: the signal is 3.9 standard deviations above the reference mean, so scale is not neutral.

In [ ]:
vectors_w = 2  # number of activation vectors.
width_w = 128  # vector length.
bytes_per_float_w = 4  # float32 storage.
mem_kb_w = vectors_w * width_w * bytes_per_float_w / 1024  # activation memory.
params_dense_w = 1_000_000  # dense parameter count.
remaining_frac_w = 0.10  # 90 percent sparse ticket.
params_ticket_w = int(params_dense_w * remaining_frac_w)  # surviving parameter count.
print("activation memory KB:", round(mem_kb_w, 3))  # 1 KB.
print("ticket parameters:", params_ticket_w, "of", params_dense_w)  # 100k of 1M.
assert round(mem_kb_w, 3) == 1.000 and params_ticket_w == 100000  # concrete bookkeeping.

▶ What you'll see: even a tiny activation block has a measurable memory cost, and 90 percent sparsity cuts a million weights to 100,000 survivors.

In [ ]:
plt.figure(figsize=(5, 3))  # visualize dense versus sparse parameter counts.
plt.bar(["dense", "10 percent ticket"], [params_dense_w, params_ticket_w], color=["gray", "teal"])  # compare counts.
plt.title("6: parameter memory pressure")  # title resource plot.
plt.ylabel("weights")  # label count axis.
plt.show()  # display bar chart.

▶ What you'll see: the sparse ticket keeps one tenth of the weights, which is the practical motivation for finding it.

*Why it's done this way:* lottery tickets sit at the intersection of optimization and capacity control. A mask is only useful if the surviving subnetwork preserves trainable signal while reducing parameter and memory cost.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Each tiny lottery-ticket toy isolates one
> computation: dense paths, local signals, masks, reset logic, pruning, or resource accounting. Every
> block prints its intermediate values, draws one visual check, and ends with an `assert`.

### ✍️ Toy 1 · Dense initialization contains candidate paths

Before pruning, each nonzero weight is a possible path that could become part of a sparse winning ticket.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)                 # seeded for reproducibility.
t1_W1 = np.array([[0.8, -0.3, 1.1, -0.6], [0.2, 0.9, -0.4, 0.7]])
t1_W2 = np.array([[0.5], [-1.0], [0.3], [0.8]])
print("first-layer weights:", t1_W1.tolist())                       # -> [[0.8, -0.3, 1.1, -0.6], [0.2, 0.9, -0.4, 0.7]]
print("second-layer weights:", t1_W2.ravel().tolist())              # -> [0.5, -1.0, 0.3, 0.8]
t1_dense_count = t1_W1.size + t1_W2.size
print("dense weight count:", t1_dense_count)                        # -> 12
print("layer shapes:", t1_W1.shape, t1_W2.shape)                    # -> (2, 4) (4, 1)
assert t1_dense_count == 12

plt.figure(figsize=(4.8, 3.0))
plt.imshow(t1_W1, cmap="coolwarm", aspect="auto")
plt.colorbar(label="initial weight")
plt.xlabel("hidden unit")
plt.ylabel("input feature")
plt.title("Toy 1 · dense first-layer paths")
plt.show()

▶ What you'll see: twelve dense weights are available before any mask decides which paths survive.

### ✍️ Toy 2 · Affine, ReLU, and softmax are local ticket arithmetic

A sparse ticket preserves local computations: weighted sums create a score, ReLU gates it, and softmax compares it to a baseline.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)                 # seeded for reproducibility.
t2_x = np.array([2.0, -1.0])
t2_w = np.array([1.0, -0.5])
t2_b = 0.25
print("input:", t2_x.tolist())                                      # -> [2.0, -1.0]
print("weights:", t2_w.tolist(), "bias:", t2_b)                    # -> [1.0, -0.5] bias 0.25
t2_z = float(t2_x @ t2_w + t2_b)
print("affine z:", round(t2_z, 3))                                  # -> 2.75
t2_h = max(0.0, t2_z)
print("ReLU h:", round(t2_h, 3))                                    # -> 2.75
t2_scores = np.array([t2_h, 0.75])
print("comparison scores:", t2_scores.tolist())                     # -> [2.75, 0.75]
t2_exp = np.exp(t2_scores - t2_scores.max())
print("shifted exponentials:", np.round(t2_exp, 3).tolist())        # -> [1.0, 0.135]
t2_prob = t2_exp / t2_exp.sum()
print("softmax probabilities:", np.round(t2_prob, 3).tolist())      # -> [0.881, 0.119]
assert round(float(t2_prob[0]), 3) == 0.881

plt.figure(figsize=(4.0, 3.0))
plt.bar(["ticket", "baseline"], t2_prob, color=["teal", "gray"])
plt.ylabel("probability")
plt.title("Toy 2 · local score wins most mass")
plt.show()

▶ What you'll see: the gated ticket score receives about `88.1%` of the two-way probability mass.

### ✍️ Toy 3 · A binary mask zeros pruned weights

The ticket keeps the original coordinate system but multiplies removed weights by zero.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)                 # seeded for reproducibility.
t3_W = np.array([[0.8, -0.2, 1.0, -0.5], [-0.7, 0.4, 0.1, 0.9]])
t3_threshold = 0.6
print("dense weights:", t3_W.tolist())                              # -> [[0.8, -0.2, 1.0, -0.5], [-0.7, 0.4, 0.1, 0.9]]
print("threshold:", t3_threshold)                                   # -> 0.6
t3_mask = (np.abs(t3_W) >= t3_threshold).astype(int)
print("mask:", t3_mask.tolist())                                    # -> [[1, 0, 1, 0], [1, 0, 0, 1]]
t3_ticket_W = t3_mask * t3_W
print("masked weights:", np.round(t3_ticket_W, 3).tolist())         # -> [[0.8, -0.0, 1.0, -0.0], [-0.7, 0.0, 0.0, 0.9]]
t3_x = np.array([1.0, -1.0])
print("input:", t3_x.tolist())                                      # -> [1.0, -1.0]
t3_dense_pre = t3_x @ t3_W
print("dense pre-activation:", np.round(t3_dense_pre, 3).tolist())  # -> [1.5, -0.6, 0.9, -1.4]
t3_ticket_pre = t3_x @ t3_ticket_W
print("ticket pre-activation:", np.round(t3_ticket_pre, 3).tolist())  # -> [1.5, 0.0, 1.0, -0.9]
assert int(t3_mask.sum()) == 4
assert np.all(t3_ticket_W[t3_mask == 0] == 0)

plt.figure(figsize=(4.8, 3.0))
plt.imshow(t3_mask, cmap="Greens", aspect="auto")
plt.colorbar(label="keep=1")
plt.xlabel("hidden unit")
plt.ylabel("input feature")
plt.title("Toy 3 · binary keep/remove mask")
plt.show()

▶ What you'll see: four edges survive, and the sparse pre-activation changes because zeros no longer carry signal.

### ✍️ Toy 4 · Reset survivors, do not keep late-pruned values

Lottery-ticket testing compares the trained mask applied to original weights against the same mask applied to trained weights.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)                 # seeded for reproducibility.
t4_initial = np.array([0.5, -1.0, 0.2, 1.5, -0.4, 0.8])
t4_grad = np.array([0.1, -0.3, 0.2, -0.4, 0.5, -0.2])
t4_eta = 0.2
print("initial weights:", t4_initial.tolist())                       # -> [0.5, -1.0, 0.2, 1.5, -0.4, 0.8]
print("gradient:", t4_grad.tolist())                                # -> [0.1, -0.3, 0.2, -0.4, 0.5, -0.2]
t4_trained = t4_initial - t4_eta * t4_grad
print("trained weights:", np.round(t4_trained, 3).tolist())         # -> [0.48, -0.94, 0.16, 1.58, -0.5, 0.84]
t4_mask = np.array([1, 0, 1, 1, 0, 1])
print("mask:", t4_mask.tolist())                                    # -> [1, 0, 1, 1, 0, 1]
t4_reset_ticket = t4_mask * t4_initial
print("reset ticket:", np.round(t4_reset_ticket, 3).tolist())       # -> [0.5, -0.0, 0.2, 1.5, -0.0, 0.8]
t4_late_pruned = t4_mask * t4_trained
print("late-pruned weights:", np.round(t4_late_pruned, 3).tolist()) # -> [0.48, -0.0, 0.16, 1.58, -0.0, 0.84]
assert t4_reset_ticket[0] == t4_initial[0]
assert t4_late_pruned[0] != t4_reset_ticket[0]

plt.figure(figsize=(5.0, 3.0))
t4_idx = np.arange(t4_initial.size)
plt.plot(t4_idx, t4_reset_ticket, marker="o", label="reset", color="teal")
plt.plot(t4_idx, t4_late_pruned, marker="x", label="late-pruned", color="crimson")
plt.xlabel("weight index")
plt.ylabel("masked value")
plt.title("Toy 4 · same mask, different values")
plt.legend()
plt.show()

▶ What you'll see: the mask is identical, but reset survivors return to their original lucky values.

### ✍️ Toy 5 · Magnitude pruning removes the smallest weights gradually

A pruning round keeps the largest magnitudes, and an iterative schedule compounds the remaining fraction.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)                 # seeded for reproducibility.
t5_abs = np.array([0.90, 0.05, 0.42, 0.12, 1.10, 0.31, 0.08, 0.70, 0.22, 0.60])
t5_keep_fraction = 0.60
print("trained magnitudes:", t5_abs.tolist())                        # -> [0.9, 0.05, 0.42, 0.12, 1.1, 0.31, 0.08, 0.7, 0.22, 0.6]
print("keep fraction:", t5_keep_fraction)                           # -> 0.6
t5_cut = np.sort(t5_abs)[int((1 - t5_keep_fraction) * t5_abs.size)]
print("magnitude cutoff:", round(float(t5_cut), 3))                 # -> 0.31
t5_mask = (t5_abs >= t5_cut).astype(int)
print("flat mask:", t5_mask.tolist())                               # -> [1, 0, 1, 0, 1, 1, 0, 1, 0, 1]
t5_rounds = np.arange(5)
print("rounds:", t5_rounds.tolist())                                # -> [0, 1, 2, 3, 4]
t5_remaining = 0.8 ** t5_rounds
print("remaining fractions:", np.round(t5_remaining, 3).tolist())   # -> [1.0, 0.8, 0.64, 0.512, 0.41]
assert int(t5_mask.sum()) == 6
assert round(float(t5_remaining[-1]), 3) == 0.41

plt.figure(figsize=(4.8, 3.0))
plt.plot(t5_rounds, t5_remaining, marker="o", color="purple")
plt.xlabel("pruning round")
plt.ylabel("fraction remaining")
plt.ylim(0, 1.05)
plt.title("Toy 5 · iterative pruning path")
plt.show()

▶ What you'll see: six of ten weights survive this round, while repeated 80% keeps shrink smoothly.

### ✍️ Toy 6 · Scale checks and sparse memory decide usefulness

A ticket is useful only if its surviving activations stay on a stable scale and its sparse parameter count actually saves memory.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)                 # seeded for reproducibility.
t6_activation = np.array([2.0, 4.0, 1.0, 3.0, 5.0, 2.0])
print("activation values:", t6_activation.tolist())                  # -> [2.0, 4.0, 1.0, 3.0, 5.0, 2.0]
t6_mean = t6_activation.mean()
print("mean:", round(float(t6_mean), 3))                             # -> 2.833
t6_var = t6_activation.var()
print("variance:", round(float(t6_var), 3))                          # -> 1.806
t6_normed = (t6_activation - t6_mean) / np.sqrt(t6_var + 1e-5)
print("normalized values:", np.round(t6_normed, 3).tolist())        # -> [-0.62, 0.868, -1.364, 0.124, 1.612, -0.62]
t6_dense_params = 12
t6_ticket_params = 4
t6_dense_bytes = t6_dense_params * 4
t6_ticket_bytes = t6_ticket_params * 4
print("dense bytes:", t6_dense_bytes)                                # -> 48
print("ticket bytes:", t6_ticket_bytes)                              # -> 16
t6_saved = 1 - t6_ticket_bytes / t6_dense_bytes
print("memory saved fraction:", round(float(t6_saved), 3))           # -> 0.667
assert abs(float(t6_normed.mean())) < 1e-12
assert round(float(t6_saved), 3) == 0.667

plt.figure(figsize=(4.4, 3.0))
plt.bar(["dense", "ticket"], [t6_dense_bytes, t6_ticket_bytes], color=["gray", "teal"])
plt.ylabel("bytes")
plt.title("Toy 6 · sparse ticket stores fewer weights")
plt.show()

▶ What you'll see: the normalized activations are centered, and the 4-weight ticket uses one third of the dense bytes.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for masks, matrix products, gradients, and small neural networks.
import matplotlib.pyplot as plt  # load Matplotlib for heatmaps, curves, and bar charts.
np.random.seed(0)  # make all examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Count dense weights before pruning

**Goal.** Count the weights in a tiny dense network, because a lottery ticket is defined by how many of those original coordinates survive. We build it in 2 steps.

In [ ]:
W1_b1 = np.array([[1.2, -0.7, 0.3, -1.1], [0.4, 1.0, -0.8, 0.6]])  # define input-to-hidden weights.
W2_b1 = np.array([[0.9], [-1.3], [0.5], [0.2]])  # define hidden-to-output weights.
print("W1 shape:", W1_b1.shape, "W2 shape:", W2_b1.shape)  # inspect layer shapes.

▶ What you'll see: two weight matrices, one with 8 entries and one with 4 entries.

In [ ]:
total_b1 = W1_b1.size + W2_b1.size  # count all trainable weights in the dense network.
print("total dense weights:", total_b1)  # inspect the pruning denominator.
assert total_b1 == 12  # concrete dense count.
plt.figure(figsize=(4, 3))  # create a compact count chart.
plt.bar(["W1", "W2"], [W1_b1.size, W2_b1.size], color="teal")  # compare layer sizes.
plt.title("Basic 1: dense weight counts")  # title the plot.
plt.ylabel("weights")  # label count axis.
plt.show()  # display the chart.

▶ What you'll see: most weights in this toy network are in the first layer.

👀 Takeaway: sparsity is measured relative to the dense network's original parameter count.

### Basic 2 — Build a binary mask

**Goal.** Create a 0/1 mask from magnitudes, because lottery tickets keep selected coordinates and zero out the rest. We build it in 2 steps.

In [ ]:
weights_b2 = np.array([1.2, -0.7, 0.3, -1.1, 0.4, 1.0, -0.8, 0.6])  # flatten one layer's weights.
threshold_b2 = 0.75  # choose a visible magnitude cutoff.
print("absolute weights:", np.round(np.abs(weights_b2), 2))  # inspect saliency scores.

▶ What you'll see: larger-magnitude weights are candidates to survive pruning.

In [ ]:
mask_b2 = (np.abs(weights_b2) >= threshold_b2).astype(int)  # keep weights above the cutoff.
print("mask:", mask_b2, "kept:", int(mask_b2.sum()))  # inspect binary selection.
assert int(mask_b2.sum()) == 4  # concrete count for this vector.
plt.figure(figsize=(5, 3))  # create a mask bar chart.
plt.bar(np.arange(weights_b2.size), mask_b2, color="purple")  # draw kept versus pruned edges.
plt.title("Basic 2: binary pruning mask")  # title the plot.
plt.ylabel("keep = 1")  # label mask value.
plt.show()  # display the chart.

▶ What you'll see: four coordinates are kept and four are pruned.

👀 Takeaway: a mask is architecture selection expressed as arithmetic multiplication.

### Basic 3 — Apply a mask to weights

**Goal.** Multiply weights by a binary mask, because `m ⊙ θ` is the subnetwork used by the lottery-ticket formula. We build it in 2 steps.

In [ ]:
W_b3 = np.array([[1.2, -0.7, 0.3, -1.1], [0.4, 1.0, -0.8, 0.6]])  # original first-layer weights.
M_b3 = (np.abs(W_b3) >= 0.75).astype(float)  # binary mask over the same shape.
print("mask shape:", M_b3.shape)  # verify shape compatibility.

▶ What you'll see: the mask has the same 2×4 shape as the weight matrix.

In [ ]:
W_masked_b3 = M_b3 * W_b3  # apply elementwise mask.
print("masked W:\n", W_masked_b3)  # inspect zeroed weights.
assert np.count_nonzero(W_masked_b3) == 4  # concrete nonzero count.
plt.figure(figsize=(5, 3))  # create a heatmap.
plt.imshow(W_masked_b3, cmap="coolwarm", aspect="auto")  # visualize surviving weights.
plt.colorbar(label="weight")  # add scale.
plt.title("Basic 3: m ⊙ W")  # title heatmap.
plt.show()  # display plot.

▶ What you'll see: pruned entries become exactly zero while surviving entries keep their original values.

👀 Takeaway: pruning changes connectivity, not the tensor shapes used by the forward pass.

### Basic 4 — Compute a sparse forward signal

**Goal.** Compare dense and masked pre-activations, because a ticket is judged by the function it computes after pruning. We build it in 3 steps.

In [ ]:
x_b4 = np.array([1.5, -0.5])  # define one input example.
W_b4 = np.array([[1.2, -0.7, 0.3, -1.1], [0.4, 1.0, -0.8, 0.6]])  # dense first layer.
M_b4 = (np.abs(W_b4) >= 0.75).astype(float)  # pruning mask.
print("input:", x_b4)  # inspect input.

▶ What you'll see: a two-feature input that will be multiplied by dense and sparse weights.

In [ ]:
z_dense_b4 = x_b4 @ W_b4  # dense pre-activation.
z_sparse_b4 = x_b4 @ (M_b4 * W_b4)  # masked pre-activation.
print("dense z:", np.round(z_dense_b4, 3))  # inspect dense signal.
print("sparse z:", np.round(z_sparse_b4, 3))  # inspect sparse signal.
assert round(float(z_sparse_b4[0]), 3) == 1.800  # concrete masked value.

▶ What you'll see: the sparse signal differs where pruned input paths used to contribute.

In [ ]:
plt.figure(figsize=(5, 3))  # create a comparison chart.
plt.plot(z_dense_b4, marker="o", label="dense")  # plot dense activations.
plt.plot(z_sparse_b4, marker="s", label="masked")  # plot sparse activations.
plt.title("Basic 4: dense vs sparse pre-activations")  # title comparison.
plt.xlabel("hidden unit")  # label hidden units.
plt.ylabel("z")  # label pre-activation value.
plt.legend()  # show curve labels.
plt.show()  # display chart.

▶ What you'll see: masking changes some hidden-unit inputs but leaves kept paths visible.

👀 Takeaway: a sparse ticket must preserve enough forward signal after many edges are removed.

### Basic 5 — Gate a signal with ReLU

**Goal.** Apply ReLU to pre-activations, because deep networks route signal through nonlinear gates as well as weights. We build it in 2 steps.

In [ ]:
z_b5 = np.array([1.8, -1.55, 0.85, -1.95])  # use sparse pre-activations from a toy pass.
print("pre-activations:", z_b5)  # inspect values before the gate.

▶ What you'll see: some hidden units are positive and some are negative.

In [ ]:
h_b5 = np.maximum(0.0, z_b5)  # ReLU keeps positive signals and zeroes negative ones.
print("ReLU activations:", h_b5)  # inspect gated output.
assert np.array_equal(h_b5, np.array([1.8, 0.0, 0.85, 0.0]))  # concrete gate result.
plt.figure(figsize=(5, 3))  # create a gate plot.
plt.bar(np.arange(4), h_b5, color="seagreen")  # show active hidden units.
plt.title("Basic 5: ReLU gates sparse paths")  # title plot.
plt.xlabel("hidden unit")  # label units.
plt.ylabel("activation")  # label activation scale.
plt.show()  # display plot.

▶ What you'll see: only hidden units with positive pre-activations pass signal forward.

👀 Takeaway: lottery tickets preserve useful paths through both weights and nonlinear gates.

### Basic 6 — Take one gradient step

**Goal.** Update a scalar parameter by gradient descent, because training a ticket still requires repeated optimizer nudges. We build it in 2 steps.

In [ ]:
theta_b6 = 2.0  # define the current scalar weight.
eta_b6 = 0.07  # define learning rate.
g_b6 = 1.8  # define gradient.
print("theta, eta, grad:", theta_b6, eta_b6, g_b6)  # inspect update ingredients.

▶ What you'll see: the step size is learning rate times gradient.

In [ ]:
theta_next_b6 = theta_b6 - eta_b6 * g_b6  # perform gradient descent.
print("next theta:", round(theta_next_b6, 3))  # inspect updated parameter.
assert round(theta_next_b6, 3) == 1.874  # concrete lesson update.
plt.figure(figsize=(4, 3))  # create before-after chart.
plt.bar(["before", "after"], [theta_b6, theta_next_b6], color=["gray", "teal"])  # compare values.
plt.title("Basic 6: one optimizer nudge")  # title plot.
plt.ylabel("theta")  # label parameter scale.
plt.show()  # display chart.

▶ What you'll see: the parameter moves downward by 0.126.

👀 Takeaway: sparse subnetworks train by the same gradient logic as dense networks, just with masked parameters.

### Basic 7 — Reset survivors to the original initialization

**Goal.** Distinguish reset tickets from late-pruned trained weights, because the hypothesis depends on the original lucky initialization. We build it in 3 steps.

In [ ]:
W0_b7 = np.array([1.2, -0.7, 0.3, -1.1])  # original initialization.
Wtrained_b7 = np.array([1.0, -0.2, 0.1, -1.5])  # pretend trained weights after dense training.
mask_b7 = np.array([1, 0, 0, 1])  # selected sparse architecture.
print("mask:", mask_b7)  # inspect selected coordinates.

▶ What you'll see: the same two coordinates will be kept in both sparse versions.

In [ ]:
reset_ticket_b7 = mask_b7 * W0_b7  # lottery-ticket reset version.
late_pruned_b7 = mask_b7 * Wtrained_b7  # compressed trained-network version.
print("reset ticket:", reset_ticket_b7)  # inspect original surviving values.
print("late pruned:", late_pruned_b7)  # inspect trained surviving values.
assert np.allclose(reset_ticket_b7, np.array([1.2, 0.0, 0.0, -1.1]))  # concrete reset vector.

▶ What you'll see: reset and late-pruned subnetworks share the mask but not the values.

In [ ]:
plt.figure(figsize=(5, 3))  # create a side-by-side comparison.
plt.plot(reset_ticket_b7, marker="o", label="reset to θ0")  # plot reset values.
plt.plot(late_pruned_b7, marker="s", label="late-pruned θT")  # plot trained values.
plt.title("Basic 7: same mask, different weights")  # title plot.
plt.legend()  # show labels.
plt.show()  # display chart.

▶ What you'll see: surviving coordinates can have noticeably different numerical values after reset.

👀 Takeaway: a winning ticket is a sparse mask plus the original initialization, not merely a pruned trained model.

### Basic 8 — Measure sparsity

**Goal.** Compute kept fraction and sparsity, because lottery-ticket claims are meaningful only with explicit compression numbers. We build it in 2 steps.

In [ ]:
mask_b8 = np.array([1, 0, 1, 0, 0, 1, 1, 0, 0, 0])  # define a toy mask.
kept_b8 = int(mask_b8.sum())  # count survivors.
total_b8 = mask_b8.size  # count original weights.
print("kept / total:", kept_b8, "/", total_b8)  # inspect mask size.

▶ What you'll see: four of ten weights survive.

In [ ]:
kept_frac_b8 = kept_b8 / total_b8  # compute retained fraction.
sparsity_b8 = 1 - kept_frac_b8  # compute pruned fraction.
print("kept fraction:", round(kept_frac_b8, 3), "sparsity:", round(sparsity_b8, 3))  # inspect compression.
assert round(sparsity_b8, 3) == 0.600  # concrete sparsity.
plt.figure(figsize=(4, 3))  # create a compression chart.
plt.bar(["kept", "pruned"], [kept_frac_b8, sparsity_b8], color=["teal", "crimson"])  # compare fractions.
plt.title("Basic 8: sparsity accounting")  # title chart.
plt.ylabel("fraction")  # label scale.
plt.show()  # display plot.

▶ What you'll see: this mask is 60 percent sparse and retains 40 percent of original weights.

👀 Takeaway: always report both accuracy behavior and the fraction of weights that remain.

### Basic 9 — Normalize a large activation

**Goal.** Compute a normalized activation, because scale drift can make an otherwise sensible ticket hard to train. We build it in 2 steps.

In [ ]:
value_b9 = 2.95  # activation value.
mean_b9 = 1.0  # reference mean.
var_b9 = 0.25  # reference variance.
print("value, mean, variance:", value_b9, mean_b9, var_b9)  # inspect normalization ingredients.

▶ What you'll see: the activation is above the reference mean.

In [ ]:
normed_b9 = (value_b9 - mean_b9) / np.sqrt(var_b9 + 1e-5)  # normalize by standard deviation.
print("normalized value:", round(float(normed_b9), 3))  # inspect standardized signal.
assert round(float(normed_b9), 3) == 3.900  # concrete lesson number.
plt.figure(figsize=(4, 3))  # create a diagnostic bar.
plt.bar(["raw", "normalized"], [value_b9, normed_b9], color=["gray", "teal"])  # compare raw and normalized scales.
plt.title("Basic 9: scale check")  # title plot.
plt.show()  # display plot.

▶ What you'll see: the normalized signal is 3.9, meaning it is unusually high relative to the reference variance.

👀 Takeaway: sparse subnetworks still need scale control so gradients stay reliable.

### Basic 10 — Compute memory saved by a sparse ticket

**Goal.** Compare dense and sparse parameter counts, because a ticket is interesting partly because it reduces cost. We build it in 2 steps.

In [ ]:
dense_params_b10 = 1_000_000  # define a dense model size.
kept_fraction_b10 = 0.10  # define a 90 percent-sparse ticket.
ticket_params_b10 = int(dense_params_b10 * kept_fraction_b10)  # compute surviving parameters.
print("dense params:", dense_params_b10, "ticket params:", ticket_params_b10)  # inspect counts.

▶ What you'll see: a 10 percent ticket keeps 100,000 of 1,000,000 weights.

In [ ]:
savings_b10 = 1 - ticket_params_b10 / dense_params_b10  # compute compression fraction.
print("parameter reduction:", f"{100 * savings_b10:.1f} percent")  # inspect savings percentage.
assert round(savings_b10, 2) == 0.90  # concrete 90 percent reduction.
plt.figure(figsize=(4, 3))  # create comparison plot.
plt.bar(["dense", "ticket"], [dense_params_b10, ticket_params_b10], color=["gray", "teal"])  # compare counts.
plt.title("Basic 10: sparse parameter budget")  # title plot.
plt.ylabel("weights")  # label count axis.
plt.show()  # display plot.

▶ What you'll see: the ticket bar is one tenth the height of the dense bar.

👀 Takeaway: pruning is a capacity and resource-control mechanism, not just a visualization trick.

## 🟡 Easy

### Easy 1 — Train a dense model before pruning

**Goal.** Fit a tiny dense linear network, because lottery-ticket pruning first needs trained weights whose magnitudes can be inspected. We build it in 4 steps.

In [ ]:
X_e1 = np.array([[1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 1.0, 0.0], [1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 1.0], [1.0, 0.5, 0.0, 0.5], [0.5, 1.0, 0.5, 0.0]])  # six examples with four candidate paths.
y_e1 = X_e1 @ np.array([2.0, -1.5, 0.0, 0.0])  # only the first two paths truly matter.
rng_e1 = np.random.default_rng(1)  # local reproducible initialization.
w_e1 = 0.1 * rng_e1.normal(size=4)  # dense initial weights.
print("initial weights:", np.round(w_e1, 3))  # inspect the random draw.

▶ What you'll see: four small random weights, two of which will become important after training.

In [ ]:
losses_e1 = []  # record mean squared error.
for step_e1 in range(300):  # run gradient descent.
    pred_e1 = X_e1 @ w_e1  # dense predictions.
    err_e1 = pred_e1 - y_e1  # residual vector.
    losses_e1.append(float(np.mean(err_e1 ** 2)))  # store loss.
    grad_e1 = (2 / X_e1.shape[0]) * X_e1.T @ err_e1  # MSE gradient.
    w_e1 = w_e1 - 0.08 * grad_e1  # update dense weights.
print("final weights:", np.round(w_e1, 3))  # inspect trained weights.
print("loss start -> end:", round(losses_e1[0], 3), "->", round(losses_e1[-1], 5))  # inspect learning.
assert losses_e1[-1] < 0.01  # concrete training success.

▶ What you'll see: the first two weights grow toward 2 and −1.5, while loss becomes very small.

In [ ]:
plt.figure(figsize=(5, 3))  # create training curve.
plt.plot(losses_e1, color="teal")  # plot dense loss.
plt.title("Easy 1: dense training loss")  # title plot.
plt.xlabel("step")  # label steps.
plt.ylabel("MSE")  # label error.
plt.show()  # display curve.

▶ What you'll see: a smooth loss decrease, giving pruning a trained model to inspect.

👀 Takeaway: lottery-ticket workflows usually select masks after some dense training has revealed which weights matter.

### Easy 2 — Prune by trained magnitude

**Goal.** Build a mask from the trained dense weights, because magnitude pruning keeps weights that ended training farthest from zero. We build it in 3 steps.

In [ ]:
trained_e2 = np.array([1.95, -1.43, 0.08, -0.04])  # representative trained weights from a dense run.
keep_e2 = 2  # keep the two largest magnitudes.
order_e2 = np.argsort(np.abs(trained_e2))[::-1]  # sort by absolute magnitude descending.
print("magnitude order:", order_e2)  # inspect which coordinates look important.

▶ What you'll see: coordinates 0 and 1 are the two largest by far.

In [ ]:
mask_e2 = np.zeros_like(trained_e2)  # start with everything pruned.
mask_e2[order_e2[:keep_e2]] = 1.0  # keep the top-k magnitudes.
pruned_e2 = mask_e2 * trained_e2  # apply the mask.
print("mask:", mask_e2.astype(int), "pruned weights:", pruned_e2)  # inspect sparse model.
assert np.array_equal(mask_e2, np.array([1.0, 1.0, 0.0, 0.0]))  # concrete mask.

▶ What you'll see: the mask selects exactly the two coordinates that generated the target.

In [ ]:
plt.figure(figsize=(5, 3))  # create magnitude plot.
plt.bar(np.arange(4), np.abs(trained_e2), color=np.where(mask_e2 > 0, "teal", "gray"))  # color kept weights.
plt.title("Easy 2: magnitude pruning")  # title plot.
plt.xlabel("weight index")  # label coordinate.
plt.ylabel("|trained weight|")  # label magnitude.
plt.show()  # display plot.

▶ What you'll see: the kept bars are visibly larger than the pruned bars.

👀 Takeaway: magnitude pruning is a simple saliency heuristic, not an oracle, so we inspect the selected mask explicitly.

### Easy 3 — Reset the mask to its original initialization

**Goal.** Train the sparse mask from the original initialization, because the lottery-ticket test is `m ⊙ θ0`, not `m ⊙ θT`. We build it in 4 steps.

In [ ]:
X_e3 = np.array([[1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 1.0, 0.0], [1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 1.0], [1.0, 0.5, 0.0, 0.5], [0.5, 1.0, 0.5, 0.0]])  # reuse toy data.
y_e3 = X_e3 @ np.array([2.0, -1.5, 0.0, 0.0])  # target depends on first two coordinates.
theta0_e3 = np.array([0.03, 0.08, 0.03, -0.13])  # saved original initialization.
mask_e3 = np.array([1.0, 1.0, 0.0, 0.0])  # mask found by magnitude pruning.
w_ticket_e3 = mask_e3 * theta0_e3  # reset surviving weights to original values.
print("reset ticket init:", np.round(w_ticket_e3, 3))  # inspect starting sparse weights.

▶ What you'll see: only the first two original weights remain nonzero.

In [ ]:
losses_e3 = []  # record sparse-ticket loss.
for step_e3 in range(300):  # train only surviving coordinates.
    pred_e3 = X_e3 @ w_ticket_e3  # sparse prediction.
    err_e3 = pred_e3 - y_e3  # residuals.
    losses_e3.append(float(np.mean(err_e3 ** 2)))  # store loss.
    grad_e3 = (2 / X_e3.shape[0]) * X_e3.T @ err_e3  # dense-form gradient.
    w_ticket_e3 = mask_e3 * (w_ticket_e3 - 0.08 * grad_e3)  # update then reapply mask.
print("trained ticket weights:", np.round(w_ticket_e3, 3))  # inspect sparse solution.
print("ticket loss:", round(losses_e3[-1], 5))  # inspect final loss.
assert losses_e3[-1] < 0.01  # concrete sparse success.

▶ What you'll see: the reset ticket trains to a low loss while pruned weights stay zero.

In [ ]:
plt.figure(figsize=(5, 3))  # create loss plot.
plt.plot(losses_e3, color="purple")  # plot ticket learning curve.
plt.title("Easy 3: reset ticket trains")  # title plot.
plt.xlabel("step")  # label optimizer steps.
plt.ylabel("MSE")  # label loss.
plt.show()  # display plot.

▶ What you'll see: the sparse reset model descends almost like the dense model because the mask kept the true paths.

👀 Takeaway: the winning-ticket claim is strongest when the original masked initialization can train well on its own.

### Easy 4 — Compare a ticket mask with a bad mask

**Goal.** Train two equally sparse masks, because sparsity alone is not enough; the particular surviving coordinates matter. We build it in 4 steps.

In [ ]:
X_e4 = np.array([[1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 1.0, 0.0], [1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 1.0], [1.0, 0.5, 0.0, 0.5], [0.5, 1.0, 0.5, 0.0]])  # toy data.
y_e4 = X_e4 @ np.array([2.0, -1.5, 0.0, 0.0])  # target.
theta0_e4 = np.array([0.03, 0.08, 0.03, -0.13])  # shared initialization.
good_mask_e4 = np.array([1.0, 1.0, 0.0, 0.0])  # keeps useful coordinates.
bad_mask_e4 = np.array([0.0, 0.0, 1.0, 1.0])  # keeps irrelevant coordinates.
print("good kept:", good_mask_e4.astype(int), "bad kept:", bad_mask_e4.astype(int))  # inspect masks.

▶ What you'll see: both masks keep two weights, but they keep different coordinates.

In [ ]:
final_losses_e4 = []  # store final loss for each mask.
curves_e4 = []  # store learning curves for plotting.
for mask_e4 in [good_mask_e4, bad_mask_e4]:  # train each sparse architecture.
    w_e4 = mask_e4 * theta0_e4.copy()  # reset to original initialization under this mask.
    losses_mask_e4 = []  # loss curve for this mask.
    for step_e4 in range(300):  # run gradient descent.
        err_e4 = X_e4 @ w_e4 - y_e4  # residuals.
        losses_mask_e4.append(float(np.mean(err_e4 ** 2)))  # store loss.
        grad_e4 = (2 / X_e4.shape[0]) * X_e4.T @ err_e4  # MSE gradient.
        w_e4 = mask_e4 * (w_e4 - 0.08 * grad_e4)  # update only kept weights.
    final_losses_e4.append(losses_mask_e4[-1])  # store final loss.
    curves_e4.append(losses_mask_e4)  # store curve.
print("final losses good vs bad:", np.round(final_losses_e4, 4))  # inspect performance gap.
assert final_losses_e4[0] < 0.01 and final_losses_e4[1] > 0.05  # concrete gap.

▶ What you'll see: the good mask learns well, while the bad mask remains far worse on this dataset.

In [ ]:
plt.figure(figsize=(5, 3))  # create mask comparison plot.
plt.plot(curves_e4[0], label="good ticket", color="teal")  # good mask curve.
plt.plot(curves_e4[1], label="bad sparse mask", color="crimson")  # bad mask curve.
plt.yscale("log")  # log scale shows both curves.
plt.title("Easy 4: sparsity is not enough")  # title plot.
plt.xlabel("step")  # label steps.
plt.ylabel("MSE, log scale")  # label error.
plt.legend()  # show labels.
plt.show()  # display plot.

▶ What you'll see: the good mask drops orders of magnitude lower than the equally sparse bad mask.

👀 Takeaway: a winning ticket is a specific sparse subnetwork, not any subnetwork with the same number of weights.

### Easy 5 — Track iterative pruning fractions

**Goal.** Simulate repeated pruning rounds, because winning-ticket searches often remove a fixed fraction at a time. We build it in 3 steps.

In [ ]:
initial_weights_e5 = 1000  # dense parameter count.
prune_each_round_e5 = 0.20  # remove 20 percent of remaining weights each round.
rounds_e5 = np.arange(6)  # include the dense model and five pruning rounds.
print("rounds:", rounds_e5)  # inspect pruning schedule indices.

▶ What you'll see: the schedule starts at round 0 before any pruning.

In [ ]:
remaining_e5 = initial_weights_e5 * ((1 - prune_each_round_e5) ** rounds_e5)  # multiplicative remaining count.
sparsity_e5 = 1 - remaining_e5 / initial_weights_e5  # pruned fraction by round.
print("remaining weights:", np.round(remaining_e5).astype(int))  # inspect counts.
print("sparsity:", np.round(sparsity_e5, 3))  # inspect compression.
assert round(float(sparsity_e5[-1]), 3) == 0.672  # concrete final sparsity.

▶ What you'll see: after five rounds, about 328 weights remain and sparsity is about 67.2 percent.

In [ ]:
plt.figure(figsize=(5, 3))  # create schedule plot.
plt.plot(rounds_e5, remaining_e5 / initial_weights_e5, marker="o", color="purple")  # show remaining fraction.
plt.title("Easy 5: iterative pruning schedule")  # title plot.
plt.xlabel("round")  # label pruning rounds.
plt.ylabel("fraction remaining")  # label remaining fraction.
plt.show()  # display curve.

▶ What you'll see: the curve decays smoothly because each round prunes a fraction of what remains.

👀 Takeaway: iterative pruning searches sparse subnetworks gradually instead of betting everything on one cutoff.

## 🔴 Advanced

### Advanced 1 — Sweep sparsity and measure trainability

**Goal.** Train masks of different sizes, because lottery-ticket behavior appears as a curve of accuracy versus sparsity. We build it in 5 steps.

In [ ]:
X_a1 = np.array([[1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 1.0, 0.0], [1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 1.0], [1.0, 0.5, 0.0, 0.5], [0.5, 1.0, 0.5, 0.0]])  # toy data.
y_a1 = X_a1 @ np.array([2.0, -1.5, 0.0, 0.0])  # target.
theta0_a1 = np.array([0.03, 0.08, 0.03, -0.13])  # original initialization.
trained_scores_a1 = np.array([1.95, 1.43, 0.08, 0.04])  # magnitude scores from dense training.
keeps_a1 = np.array([4, 3, 2, 1])  # number of weights to keep.
print("keep counts:", keeps_a1)  # inspect sweep.

▶ What you'll see: the sweep moves from dense to extremely sparse.

In [ ]:
loss_by_keep_a1 = []  # store final train loss.
for keep_a1 in keeps_a1:  # evaluate each sparsity level.
    order_a1 = np.argsort(trained_scores_a1)[::-1]  # rank by magnitude.
    mask_a1 = np.zeros(4)  # start fully pruned.
    mask_a1[order_a1[:keep_a1]] = 1.0  # keep top coordinates.
    w_a1 = mask_a1 * theta0_a1.copy()  # reset selected coordinates.
    for step_a1 in range(300):  # train sparse model.
        err_a1 = X_a1 @ w_a1 - y_a1  # residuals.
        grad_a1 = (2 / X_a1.shape[0]) * X_a1.T @ err_a1  # gradient.
        w_a1 = mask_a1 * (w_a1 - 0.08 * grad_a1)  # masked update.
    loss_by_keep_a1.append(float(np.mean((X_a1 @ w_a1 - y_a1) ** 2)))  # final loss.
print("loss by keep:", np.round(loss_by_keep_a1, 4))  # inspect trainability.
assert loss_by_keep_a1[2] < 0.01 and loss_by_keep_a1[-1] > 0.1  # two weights good, one weight not enough.

▶ What you'll see: keeping two or more useful weights trains well, but one kept coordinate underfits.

In [ ]:
sparsity_a1 = 1 - keeps_a1 / 4  # compute sparsity for each mask.
plt.figure(figsize=(5, 3))  # create sparsity curve.
plt.plot(sparsity_a1, loss_by_keep_a1, marker="o", color="crimson")  # plot final loss.
plt.yscale("log")  # use log scale for small losses.
plt.title("Advanced 1: trainability vs sparsity")  # title plot.
plt.xlabel("sparsity")  # label sparsity axis.
plt.ylabel("final MSE, log scale")  # label loss.
plt.show()  # display plot.

▶ What you'll see: the curve stays low until pruning removes a necessary coordinate.

👀 Takeaway: the useful question is not “can we prune?” but “how far can this particular ticket be pruned before trainability breaks?”

### Advanced 2 — Compare reset, late-pruned, and random reinitialization

**Goal.** Separate three sparse-training stories, because lottery tickets specifically emphasize resetting survivors to their original values. We build it in 5 steps.

In [ ]:
X_a2 = np.array([[1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 1.0, 0.0], [1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 1.0], [1.0, 0.5, 0.0, 0.5], [0.5, 1.0, 0.5, 0.0]])  # toy data.
y_a2 = X_a2 @ np.array([2.0, -1.5, 0.0, 0.0])  # target.
mask_a2 = np.array([1.0, 1.0, 0.0, 0.0])  # winning mask.
theta0_a2 = np.array([0.03, 0.08, 0.03, -0.13])  # original draw.
thetaT_a2 = np.array([1.95, -1.43, 0.08, -0.04])  # trained dense values.
print("mask:", mask_a2.astype(int))  # inspect selected sparse architecture.

▶ What you'll see: all three variants will use the same two surviving coordinates.

In [ ]:
starts_a2 = [mask_a2 * theta0_a2, mask_a2 * thetaT_a2, mask_a2 * np.array([-0.09, 0.02, 0.11, 0.01])]  # reset, late-pruned, random.
labels_a2 = ["reset θ0", "late θT", "random reset"]  # labels for comparison.
final_losses_a2 = []  # store final losses.
for start_a2 in starts_a2:  # train each variant.
    w_a2 = start_a2.copy()  # initialize sparse weights.
    for step_a2 in range(120):  # train for a shorter budget.
        err_a2 = X_a2 @ w_a2 - y_a2  # residuals.
        grad_a2 = (2 / X_a2.shape[0]) * X_a2.T @ err_a2  # gradient.
        w_a2 = mask_a2 * (w_a2 - 0.08 * grad_a2)  # masked update.
    final_losses_a2.append(float(np.mean((X_a2 @ w_a2 - y_a2) ** 2)))  # final loss.
print("final losses:", dict(zip(labels_a2, np.round(final_losses_a2, 5))))  # inspect comparison.
assert max(final_losses_a2) < 0.05  # all can train on this easy convex toy.

▶ What you'll see: on this simple convex toy all starts learn, which warns us not to overclaim from one demo.

In [ ]:
plt.figure(figsize=(5, 3))  # create comparison bars.
plt.bar(labels_a2, final_losses_a2, color=["teal", "gray", "orange"])  # compare final losses.
plt.title("Advanced 2: same mask, different starts")  # title plot.
plt.ylabel("final MSE")  # label loss scale.
plt.xticks(rotation=15)  # rotate labels.
plt.show()  # display plot.

▶ What you'll see: the same mask can be evaluated under multiple starting-value assumptions.

👀 Takeaway: in nonconvex deep networks, reset-to-θ0 can matter; simple convex demos help define the comparison but do not prove the full hypothesis.

### Advanced 3 — Show a learning-rate stability failure

**Goal.** Train the same sparse ticket with two learning rates, because a good mask can still fail if optimizer steps are too large. We build it in 4 steps.

In [ ]:
X_a3 = np.array([[1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 1.0, 0.0], [1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 1.0], [1.0, 0.5, 0.0, 0.5], [0.5, 1.0, 0.5, 0.0]])  # toy data.
y_a3 = X_a3 @ np.array([2.0, -1.5, 0.0, 0.0])  # target.
mask_a3 = np.array([1.0, 1.0, 0.0, 0.0])  # sparse ticket mask.
rates_a3 = [0.08, 1.20]  # stable and intentionally too-large learning rates.
print("learning rates:", rates_a3)  # inspect comparison.

▶ What you'll see: one rate is modest and one is deliberately aggressive.

In [ ]:
curves_a3 = []  # store loss curves.
for eta_a3 in rates_a3:  # train with each rate.
    w_a3 = mask_a3 * np.array([0.03, 0.08, 0.03, -0.13])  # reset ticket.
    losses_a3 = []  # loss curve for this rate.
    for step_a3 in range(80):  # short training loop.
        err_a3 = X_a3 @ w_a3 - y_a3  # residuals.
        losses_a3.append(float(np.mean(err_a3 ** 2)))  # store loss.
        grad_a3 = (2 / X_a3.shape[0]) * X_a3.T @ err_a3  # gradient.
        w_a3 = mask_a3 * (w_a3 - eta_a3 * grad_a3)  # masked update.
        w_a3 = np.clip(w_a3, -20, 20)  # keep unstable demo finite.
    curves_a3.append(losses_a3)  # save curve.
print("final losses:", [round(c[-1], 3) for c in curves_a3])  # inspect stability.
assert curves_a3[0][-1] < 0.01 and curves_a3[1][-1] > 1.0  # stable versus unstable.

▶ What you'll see: the modest rate converges, while the large rate remains high or oscillatory.

In [ ]:
plt.figure(figsize=(5, 3))  # create stability plot.
plt.plot(curves_a3[0], label="η=0.08", color="teal")  # stable curve.
plt.plot(curves_a3[1], label="η=1.20", color="crimson")  # unstable curve.
plt.yscale("log")  # log scale makes the gap readable.
plt.title("Advanced 3: optimizer stability")  # title plot.
plt.xlabel("step")  # label steps.
plt.ylabel("MSE, log scale")  # label loss.
plt.legend()  # show labels.
plt.show()  # display plot.

▶ What you'll see: the ticket is not magic; optimizer scale can still make training fail.

👀 Takeaway: masks, initialization, learning rate, and normalization form one training system.

### Advanced 4 — Use normalization to fix scale before pruning

**Goal.** Compare raw and standardized features, because magnitude scores and gradients can be distorted when input scales differ. We build it in 4 steps.

In [ ]:
X_raw_a4 = np.array([[10.0, 0.0], [12.0, 1.0], [9.0, -1.0], [11.0, 0.5], [8.0, -0.5]])  # first feature has larger scale.
y_a4 = 0.2 * X_raw_a4[:, 0] - 1.5 * X_raw_a4[:, 1]  # target uses both features.
means_a4 = X_raw_a4.mean(axis=0)  # feature means.
stds_a4 = X_raw_a4.std(axis=0) + 1e-8  # feature standard deviations.
print("feature stds:", np.round(stds_a4, 3))  # inspect scale mismatch.

▶ What you'll see: the first feature has a much larger raw standard deviation.

In [ ]:
X_std_a4 = (X_raw_a4 - means_a4) / stds_a4  # standardize feature columns.
print("standardized mean:", np.round(X_std_a4.mean(axis=0), 3))  # check zero mean.
print("standardized std:", np.round(X_std_a4.std(axis=0), 3))  # check unit scale.
assert np.allclose(np.round(X_std_a4.std(axis=0), 3), np.ones(2))  # concrete normalization check.

▶ What you'll see: both standardized features have mean 0 and standard deviation 1.

In [ ]:
grad_raw_a4 = (2 / X_raw_a4.shape[0]) * X_raw_a4.T @ (X_raw_a4 @ np.zeros(2) - y_a4)  # initial raw gradient.
grad_std_a4 = (2 / X_std_a4.shape[0]) * X_std_a4.T @ (X_std_a4 @ np.zeros(2) - y_a4)  # initial standardized gradient.
print("raw gradient magnitudes:", np.round(np.abs(grad_raw_a4), 3))  # inspect raw scale effect.
print("std gradient magnitudes:", np.round(np.abs(grad_std_a4), 3))  # inspect normalized scale.

▶ What you'll see: raw gradients are dominated by feature scale, while standardized gradients are easier to compare.

In [ ]:
plt.figure(figsize=(5, 3))  # create gradient comparison plot.
plt.bar(["raw f0", "raw f1", "std f0", "std f1"], np.r_[np.abs(grad_raw_a4), np.abs(grad_std_a4)], color=["crimson", "crimson", "teal", "teal"])  # compare magnitudes.
plt.title("Advanced 4: scale affects pruning signals")  # title plot.
plt.ylabel("|initial gradient|")  # label gradient scale.
plt.xticks(rotation=15)  # rotate labels.
plt.show()  # display plot.

▶ What you'll see: normalization changes the numerical evidence a pruning rule might see.

👀 Takeaway: before trusting magnitude or gradient scores, check whether scale made one coordinate look important for the wrong reason.

### Advanced 5 — Avoid pruning all paths to a hidden unit

**Goal.** Detect dead hidden units after masking, because pruning can remove every incoming or outgoing edge for a unit and silently reduce capacity. We build it in 4 steps.

In [ ]:
W1_a5 = np.array([[1.2, -0.7, 0.3, -1.1], [0.4, 1.0, -0.8, 0.6]])  # input-to-hidden weights.
W2_a5 = np.array([[0.9], [-1.3], [0.5], [0.2]])  # hidden-to-output weights.
M1_a5 = np.array([[1, 0, 0, 1], [0, 0, 0, 1]], dtype=float)  # mask leaves hidden unit 1 with no inputs.
M2_a5 = np.array([[1], [1], [0], [1]], dtype=float)  # output mask prunes hidden unit 2 output.
print("M1:\n", M1_a5.astype(int))  # inspect incoming mask.

▶ What you'll see: some hidden-unit columns have no surviving input edges.

In [ ]:
incoming_a5 = M1_a5.sum(axis=0)  # count kept incoming edges per hidden unit.
outgoing_a5 = M2_a5[:, 0]  # kept outgoing edge per hidden unit.
alive_a5 = (incoming_a5 > 0) & (outgoing_a5 > 0)  # hidden unit must have input and output paths.
print("incoming counts:", incoming_a5.astype(int))  # inspect input connectivity.
print("outgoing kept:", outgoing_a5.astype(int))  # inspect output connectivity.
print("alive hidden units:", alive_a5.astype(int))  # inspect usable units.
assert np.array_equal(alive_a5.astype(int), np.array([1, 0, 0, 1]))  # concrete alive pattern.

▶ What you'll see: two hidden units are effectively dead because at least one side of their path is missing.

In [ ]:
plt.figure(figsize=(5, 3))  # create connectivity plot.
plt.bar(np.arange(4) - 0.15, incoming_a5, width=0.3, label="incoming")  # incoming counts.
plt.bar(np.arange(4) + 0.15, outgoing_a5, width=0.3, label="outgoing")  # outgoing indicators.
plt.title("Advanced 5: hidden-unit path check")  # title plot.
plt.xlabel("hidden unit")  # label unit.
plt.ylabel("kept edges")  # label connectivity.
plt.legend()  # show labels.
plt.show()  # display plot.

▶ What you'll see: alive units need at least one kept edge on both sides; otherwise signal cannot pass through them.

👀 Takeaway: pruning masks should be inspected structurally, not just by global sparsity percentage.